In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import statsmodels.api as sm

# Load the dataset
file_path = "NYPD_Vehicle_Stop_Reports.xlsx"
df = pd.read_excel(file_path)

# Convert categorical binary flags to numeric (True/False → 1/0)
binary_cols = ['VEH_SEIZED_FLG', 'VEH_SEARCHED_FLG', 'VEH_SEARCH_CONSENT_FLG',
               'VEH_CHECKPOINT_FLG', 'FORCE_USED_FLG', 'ARREST_MADE_FLG', 'SUMMON_ISSUED_FLG']

for col in binary_cols:
    df[col] = df[col].map({'Y': 1, 'N': 0, True: 1, False: 0})

# Convert age column to numeric, handling unknown values
df['RPTED_AGE'] = pd.to_numeric(df['RPTED_AGE'], errors='coerce')

# Select relevant numerical columns
numeric_cols = binary_cols + ['RPTED_AGE', 'X_COORD_CD', 'Y_COORD_CD']
df_numeric = df[numeric_cols].dropna()

# Define the features (X) and target (y)
X = df_numeric.drop(columns=['VEH_SEARCHED_FLG'])  # Features
y = df_numeric['VEH_SEARCHED_FLG']  # Target

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train a logistic regression model
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

# Predict on the test data
y_pred = log_reg.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nConfusion Matrix:\n", conf_matrix)
print("\nClassification Report:\n", class_report)

# Visualize correlation with heatmap (already done in previous code)
correlation_matrix = df_numeric.corr()
plt.figure(figsize=(10, 6))
sns.heatmap(correlation_matrix[['VEH_SEARCHED_FLG']].sort_values(by='VEH_SEARCHED_FLG', ascending=False),
            annot=True, cmap='coolwarm', linewidths=0.5)
plt.title("Correlation of Factors with Vehicle Searches")
plt.show()

# Statsmodels for detailed logistic regression summary
X_sm = sm.add_constant(X)  # Adding constant to the model (intercept)
logit_model = sm.Logit(y, X_sm)
result = logit_model.fit()

print(result.summary())